# 04. Constraint Problems: N-Queens, Word Search, Sudoku

Place or find things while obeying rules. Most of the work is **checking constraints fast** so we can prune early.

**Learning order:** N-QUEENS → WORD SEARCH → PALINDROME PARTITIONING → RESTORE IP → SUDOKU → INTERVIEW THINKING

## 1. N-Queens

Place `n` queens on an `n × n` board, no two attacking each other (same row, column or diagonal).

**One queen per row:** loop over rows; for each row choose a **column**. Check conflicts in O(1) using sets:
- `cols`: used columns
- `diag1`: `r - c` (all cells on one `\` diagonal share it)
- `diag2`: `r + c` (all cells on one `/` diagonal share it)

In [1]:
def solve_n_queens(n):
    res, cols_of_row = [], []
    cols, d1, d2 = set(), set(), set()
    def backtrack(r):
        if r == n:
            res.append(["." * c + "Q" + "." * (n - c - 1) for c in cols_of_row])
            return
        for c in range(n):
            if c in cols or (r - c) in d1 or (r + c) in d2:
                continue                                  # prune
            cols.add(c); d1.add(r - c); d2.add(r + c); cols_of_row.append(c)
            backtrack(r + 1)
            cols.remove(c); d1.remove(r - c); d2.remove(r + c); cols_of_row.pop()
    backtrack(0)
    return res

sol = solve_n_queens(4)
print(len(sol), sol[0])   # 2 ['.Q..', '...Q', 'Q...', '..Q.']
print(len(solve_n_queens(8)))   # 92

2 ['.Q..', '...Q', 'Q...', '..Q.']
92


## 2. Word Search (grid backtracking)

Find `word` in a grid moving up/down/left/right, using each cell **at most once**. At each cell: match the next letter, **mark the cell visited** (temporarily replace it with `#`), explore the 4 neighbours, then **restore** the letter. Real life: a word puzzle.

Time O(rows · cols · 4^L) in the worst case, where `L` = word length. Pruning: stop as soon as a letter does not match.

In [2]:
def word_search(board, word):
    rows, cols = len(board), len(board[0])
    def dfs(r, c, i):
        if i == len(word):
            return True
        if not (0 <= r < rows and 0 <= c < cols) or board[r][c] != word[i]:
            return False
        saved, board[r][c] = board[r][c], "#"          # choose (mark visited)
        found = (dfs(r + 1, c, i + 1) or dfs(r - 1, c, i + 1) or
                 dfs(r, c + 1, i + 1) or dfs(r, c - 1, i + 1))
        board[r][c] = saved                            # un-choose
        return found
    return any(dfs(r, c, 0) for r in range(rows) for c in range(cols))

grid = [["A", "B", "C", "E"], ["S", "F", "C", "S"], ["A", "D", "E", "E"]]
print(word_search(grid, "ABCCED"), word_search(grid, "SEE"), word_search(grid, "ABCB"))   # True True False

True True False


## 3. Palindrome Partitioning

Split a string into pieces where **every piece is a palindrome**. At each position, try every end index; continue only if `s[start:end]` is a palindrome (prune).

In [3]:
def partition_palindromes(s):
    res, path = [], []
    def backtrack(start):
        if start == len(s):
            res.append(path[:])
            return
        for end in range(start + 1, len(s) + 1):
            piece = s[start:end]
            if piece == piece[::-1]:                   # prune non-palindromes
                path.append(piece)
                backtrack(end)
                path.pop()
    backtrack(0)
    return res

print(partition_palindromes("aab"))   # [['a', 'a', 'b'], ['aa', 'b']]

[['a', 'a', 'b'], ['aa', 'b']]


## 4. Restore IP Addresses

Split a digit string into **4 parts**, each 0 to 255, no leading zeros (except `"0"` itself). Choose a part length 1 to 3. Prune when the part is invalid or when too many/few digits remain.

In [4]:
def restore_ip_addresses(s):
    res, parts = [], []
    def backtrack(start):
        if len(parts) == 4:
            if start == len(s):
                res.append(".".join(parts))
            return
        for length in (1, 2, 3):
            if start + length > len(s):
                break
            piece = s[start:start + length]
            if (len(piece) > 1 and piece[0] == "0") or int(piece) > 255:
                continue
            parts.append(piece)
            backtrack(start + length)
            parts.pop()
    backtrack(0)
    return res

print(restore_ip_addresses("25525511135"))   # ['255.255.11.135', '255.255.111.35']
print(restore_ip_addresses("0000"))          # ['0.0.0.0']

['255.255.11.135', '255.255.111.35']
['0.0.0.0']


## 5. Sudoku Solver

Find the first empty cell, try digits `1..9` that do not conflict with the row, column or 3×3 box, recurse. If no digit works, **return False** so the previous cell changes its digit. Keep row/column/box **sets** for O(1) checks.

In [5]:
def solve_sudoku(board):
    rows = [set() for _ in range(9)]
    cols = [set() for _ in range(9)]
    boxes = [set() for _ in range(9)]
    empties = []
    for r in range(9):
        for c in range(9):
            v = board[r][c]
            if v == ".":
                empties.append((r, c))
            else:
                rows[r].add(v); cols[c].add(v); boxes[r // 3 * 3 + c // 3].add(v)

    def backtrack(i):
        if i == len(empties):
            return True
        r, c = empties[i]
        b = r // 3 * 3 + c // 3
        for d in "123456789":
            if d in rows[r] or d in cols[c] or d in boxes[b]:
                continue
            board[r][c] = d
            rows[r].add(d); cols[c].add(d); boxes[b].add(d)
            if backtrack(i + 1):
                return True
            board[r][c] = "."                          # undo
            rows[r].remove(d); cols[c].remove(d); boxes[b].remove(d)
        return False

    return backtrack(0)

puzzle = ["53..7....", "6..195...", ".98....6.", "8...6...3", "4..8.3..1",
          "7...2...6", ".6....28.", "...419..5", "....8..79"]
board = [list(row) for row in puzzle]
print(solve_sudoku(board))   # True
print(all(sorted(row) == list("123456789") for row in board))   # True

True
True


## 6. Interview thinking

**Say:** "I place one item per step and check constraints in O(1) with sets, so invalid branches die immediately."

**Common mistakes:** forgetting to **restore** the board in Word Search, wrong diagonal keys (`r - c`, `r + c`), not breaking on invalid IP parts, rebuilding the full board check at every step (O(n) instead of O(1)), forgetting to return `False` when no digit fits in Sudoku.

## Quick Revision

- N-Queens: one queen per row, sets for `col`, `r-c`, `r+c`.
- Word search: mark `#`, explore 4 directions, restore.
- Palindrome partition: try every end, continue only if palindrome.
- IP: 4 parts, 1 to 3 digits, ≤ 255, no leading zero.
- Sudoku: sets for row/col/box, return `False` to backtrack.